In [1]:
"""
Sentence-per-Paragraph Standard Deviation Module

Provides lightweight sentence distribution statistics for LLM
responses using newline-based paragraph segmentation.

Supports:
    - Regex-based sentence segmentation
    - Newline-based paragraph segmentation
    - Population standard deviation of sentence counts per paragraph
    - Robust handling of None / NaN / empty inputs

This feature is useful for:
    - Measuring document organization
    - Quantifying paragraph structure
    - Comparing structural consistency across model outputs
"""

from typing import Any
import math
import re


# ---------------------------------------------------------
# Utility: NaN-safe checking
# ---------------------------------------------------------
def _is_nan(value: Any) -> bool:
    """Return True if the input should be treated as missing."""
    if value is None:
        return True
    if isinstance(value, float):
        return math.isnan(value)
    return False


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
_PARAGRAPH_SPLIT_RE = re.compile(r"\n+")
_SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+|\n+")


# ---------------------------------------------------------
# Main Feature Extractor
# ---------------------------------------------------------
def sd_sent_per_paragraph(text: Any) -> float:
    """
    Compute the population standard deviation of sentence counts
    across paragraphs.

    Parameters
    ----------
    text : Any
        Input response text.

    Returns
    -------
    float
        Population standard deviation of sentence counts per paragraph.
    """

    # -------------------------
    # Input validation
    # -------------------------
    if _is_nan(text):
        return 0.0

    clean_text = str(text).strip()

    if not clean_text:
        return 0.0

    if clean_text.lower() == "nan":
        return 0.0

    # -------------------------
    # Paragraph Segmentation
    # -------------------------
    paragraphs = [
        paragraph.strip()
        for paragraph in _PARAGRAPH_SPLIT_RE.split(clean_text)
        if paragraph.strip()
    ]

    # -------------------------
    # Sentence Count Calculation
    # -------------------------
    sentence_counts = []

    for paragraph in paragraphs:
        sentences = [
            sentence.strip()
            for sentence in _SENTENCE_SPLIT_RE.split(paragraph)
            if sentence.strip()
        ]
        sentence_counts.append(len(sentences))

    if len(sentence_counts) <= 1:
        return 0.0

    # -------------------------
    # Statistics
    # -------------------------
    mean_value = sum(sentence_counts) / len(sentence_counts)

    variance = sum(
        (count - mean_value) ** 2
        for count in sentence_counts
    ) / len(sentence_counts)

    return float(math.sqrt(variance))


__all__ = ["sd_sent_per_paragraph"]